# Accelerated Backtesting

## Overview
There is a trade-off between the accuracy and speed of backtesting. *hftbacktest* provides highly accurate results, but it is relatively slow, making it challenging to quickly test new ideas or optimize parameters through rapid iteration. To improve backtest speed, this approach excludes certain conditions, sacrificing a certain degree of accuracy.

The main performance gain comes from precomputing fill conditions for a given running interval. This includes ignoring both fills in the queue position and the order response latency, enabling backtesting within a single loop iteration. By removing queue position estimation, we eliminate the need to fully replay the market depth feed or process every depth update.

Thus, this approach accounts for feed latency and order entry latency, but not order-response latency. In the fill simulation, queue position is not modeled, so partial fills do not occur—orders are either fully filled when crossed or not filled at all.

While these simplifications can result in a loss of accuracy—particularly in case that queue position fills are critical typically due to large tick sizes—they offer substantial performance gains.

## Runnable Tardis Test

This notebook executes the corresponding experiment through the shared
`tutorial_reproduction` runner. It uses existing Tardis files only and
does not require a Tardis API key or download data.

Defaults:

- amdserver: `/home/molly/data/tardis/binance-futures`, `2025-08-01`
- Mac: `~/Documents/tardis`, `2025-01-01`
- Window: `300` seconds

Optional environment overrides:

- `HFTBACKTEST_TARDIS_ROOT`
- `HFTBACKTEST_TARDIS_DATE`
- `HFTBACKTEST_NOTEBOOK_SECONDS`
- `HFTBACKTEST_NOTEBOOK_OUTPUT`

Active experiment: `Accelerated Backtesting.ipynb` (`accelerated_backtesting`).

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
search_roots = []
for base in (cwd, *cwd.parents):
    search_roots.extend((base, base / 'examples'))
examples_root = next(
    path for path in search_roots
    if (path / 'tutorial_reproduction').is_dir()
)
if str(examples_root) not in sys.path:
    sys.path.insert(0, str(examples_root))

from tutorial_reproduction.notebook_support import (
    context_dict,
    notebook_context,
    run_notebook_experiment,
)

In [ ]:
context = notebook_context()
context_dict(context)

In [ ]:
manifest = run_notebook_experiment('accelerated_backtesting', context)
manifest['result']

In [ ]:
assert manifest['result']['status'] != 'failed'
print('notebook:', manifest['notebook'])
print('status:', manifest['result']['status'])
print('output:', context.output_root)

## Original Tutorial Reference

The original tutorial narrative and code are retained below for comparison.
Original code cells are rendered as non-executing references so that
`Run All` remains reproducible with the configured Tardis dataset.

## Fill Conditions

- A **buy** order is eligible to fill when **order price >= best ask** or **order price > sell trade price**.  
- A **sell** order is eligible to fill when **order price <= best bid** or **order price < buy trade price**.

Because queue position is not considered, equality does not count:

- buy price == sell trade price → **not filled**
- sell price == buy trade price → **not filled**

In short, if the market price crosses the order price (strictly), the order is considered filled.

## Limit Order Fill Prices
For a given time interval [t_i, t_{i+1}]:

**Bid fill price (applies to buy orders):**
```
bid fill price = min(
    lowest best ask at the exchange over the interval,
    (lowest sell trade price + one tick) at the exchange over the interval
)
```
An open buy order with **order price >= bid fill price** is considered filled in the interval.  

**Ask fill price (applies to sell orders):**

```
ask fill price = max(
    highest best bid at the exchange over the interval,
    (highest buy trade price - one tick) at the exchange over the interval
)
```
An open sell order with **order price <= ask fill price** is considered filled in the interval.

## Order Response Latency
To maintain a single state (no separate local and exchange state) and utilize a single-loop iteration, this don't account for order response latency. Consequently, all state changes—order acceptance, cancellation, fills, and position updates—are reflected immediately on the local side.

## Preprocessed Data Structure

```
                          row[t]                                             row[t+1]
Local
+-------------------------------------------------------------+----------------------------
|local_ts[t]                                                  |local_ts[t+1]
|                                                             |
|best_bid[t]                                                  |best_bid[t+1]
|best_ask[t]                                                  |best_ask[t+1]
+-------------------------------------------------------------+---------------------------
Exchange
+-------------------------------------------------------------+---------------------------
|                                                bid_fill[t+1]|
|                                                ask_fill[t+1]|
+----------------------------+--------------------------------+---------------------------
|  order entry latency at    |order_ack_ts[t]                 |
|                local_ts[t] |                                |
|                            |best_bid_ack[t]                 |
|                            |best_ask_ack[t]                 |
|                            |                                |
|             bid_fill_ack[t]|           bid_fill_after_ack[t]|
|             ask_fill_ack[t]|           ask_fill_after_ack[t]|
+----------------------------+--------------------------------+---------------------------
```

The open order which is already acknowledged by the exchange between local_ts[t] to local_ts[t + 1] is filled by bid_fill[t + 1] or ask_fill[t + 1] based on the fill condition we describe above and the local knows it at local_ts[t + 1].

If a user sends a new order at local_ts[t], this order is checked if accepted based on best_bid_ack[t] or best_ask_ack[t] (if it is GTX, or marketable order). And if the limit order is accepted, it is checked if is filled until local_ts[t + 1] by bid_fill_after_ack[t] or ask_fill_after_ack[t]. A user know it at local_ts[t + 1].

If a user sends a cancel order at local_ts[t], the open order is checked if it is filled before cancel request is acknowledge by bid_fill_ack[t] or ask_fill_ack[t]. And also a user know it at local_ts[t + 1].

If order entry latency is high enough that the order request arrives at exchange after local_ts[t + 1], compose the row as shown in the following. 

```
Local
+------------------------------+-------------------------------------------------------------+-------------
|local_ts[t]                   |local_ts[t+1]                                                |local_ts[t+2]
|                              |                                                             |
|best_bid[t]                   |best_bid[t+1]                                                |
|best_ask[t]                   |best_ask[t+1]                                                |
+------------------------------+-------------------------------------------------------------+-------------
Exchange
+------------------------------+-------------------------------------------------------------+-------------
|                 bid_fill[t+1]|                                                bid_fill[t+2]|
|                 ask_fill[t+1]|                                                ask_fill[t+2]|
+------------------------------+------------------------+------------------------------------+-------------
|           order entry latency at local_ts[t]          |order_ack_ts[t]                     |
|                                                       |                                    |
|                                                       |best_bid_ack[t]                     |
|                                                       |best_ask_ack[t]                     |
|                                                       |                                    |
|                                        bid_fill_ack[t]|               bid_fill_after_ack[t]|
|                                        ask_fill_ack[t]|               ask_fill_after_ack[t]|
+------------------------------+------------------------+---+--------------------------------+-------------
|                              |                            |order_ack_ts[t+1]               |
|                              |                            |                                |
|                              |                            |best_bid_ack[t+1]               |
|                              |                            |best_ask_ack[t+1]               |
+------------------------------+----------------------------+--------------------------------+-------------
|                              | order entry latency at     |order_ack_ts[t+1]               |
|                              |              local_ts[t+1] |                                |
|                              |                            |best_bid_ack[t+1]               |
|                              |                            |best_ask_ack[t+1]               |
|                              |                            |                                |
|                              |           bid_fill_ack[t+1]|         bid_fill_after_ack[t+1]|
|                              |           ask_fill_ack[t+1]|         ask_fill_after_ack[t+1]|
+------------------------------+----------------------------+--------------------------------+-------------
```

## Preprocessing Market Data for Accelerated Backtesting

## Preprocessing Example

Tardis.dev provides free sample data for the first day of each month.

## Accelerated Backtesting Using Preprocessed Market Data

## Comparison of Backtesting Results with Full Backtesting

Firstly, accelerated backtest: 416 ms; full backtest: 1 min 49 s — roughly 260× faster.

You can see that there are differences in the detailed numbers, but overall, the results show similar characteristics in terms of position and equity. The differences can become larger depending on the strategy’s characteristics—especially, as mentioned earlier, for assets where fills in the queue is crucial, such as those with a large tick size. Therefore, it is still important to verify the results from accelerated backtesting against those from full backtesting.

## Precompute Signal - Order Book Imbalance

Similarly, precomputing the signal speeds up backtesting and allows rapid iteration to test ideas and tune parameters.

## Accelerated Backtesting - Order Book Imbalance

Using the precomputed order book imbalance, we perform accelerated backtesting. The only modification is that quoting based on `mid_tick` is replaced with `fair_px_tick`. 

$VAMP_{effective}$ is selected for demonstration purposes. As illustrated in [Market Making with Alpha - Order Book Imbalance](https://hftbacktest.readthedocs.io/en/latest/tutorials/Market%20Making%20with%20Alpha%20-%20Order%20Book%20Imbalance.html), there are many different ways to compute order book imbalance. The key is to identify which formulation provides the stronger signal and which parameters lead to better performance, as these may vary depending on the strategy and how the signal is monetized.

At the end of this section, you can observe that performance improves compared to `mid_tick`; however, it is ultimately necessary to evaluate results over longer periods. Such comparisons and analyses require repeated, iterative backtesting — made feasible through this accelerated backtesting framework.

In the next tutorial, we will explore a more generalized framework to forecasting and fair value pricing in research based on [A Comprehensive Framework for Pricing Models](https://hftbacktest.readthedocs.io/en/latest/tutorials/Market%20Making%20with%20Alpha%20-%20APT.html#A-Comprehensive-Framework-for-Pricing-Models).